# 03 — Thí Nghiệm Phân Tách Vòng Lặp QRS Loop & T Loop (VCG Delineation)

> **Mục tiêu nghiên cứu:**
> 1. Trực quan hóa và giải quyết bài toán cốt lõi: **Làm thế nào để phân tách chính xác Vòng QRS và Vòng T trong không gian 3D VCG?**
> 2. Khắc phục các thách thức y sinh: Đoạn ST chênh (ST elevation/depression), sóng T dẹt, điện thế thấp ở bệnh nhân nhồi máu cơ tim (MI).
> 3. Cơ chế giải thuật 2 pha thích nghi:
>    * **Vận tốc biến thiên không gian $v(t) = \|\frac{d\mathbf{v}}{dt}\|$** để bắt điểm thắt cổ chai J-point ($QRS_{off}$).
>    * **Độ lớn vector $M(t)$** để bám theo bao hình sóng T.
>    * **Hàng rào sinh lý bảo vệ (Physiological Constraints)** đảm bảo không bao giờ lỗi mảng khi gặp ca bệnh nặng.
>    * **Trực quan hóa vòng lặp 3D tách rời & Chiếu PCA 2D**.


In [ ]:
# A. THƯ VIỆN & CẤU HÌNH TRỰC QUAN HÓA
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.signal import savgol_filter, butter, sosfiltfilt, find_peaks
from sklearn.decomposition import PCA
import wfdb

# Thiết lập phong cách đồ họa chuẩn học thuật
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['figure.autolayout'] = False

DATA_ROOT = Path("content/ptbdb/physionet.org/files/ptbdb/1.0.0")
print("DATA_ROOT tồn tại:", DATA_ROOT.exists())


In [ ]:
# B. CÁC HÀM TIỀN XỬ LÝ & TẠO NHỊP ĐẠI DIỆN TRUNG VỊ (MEDIAN REPRESENTATIVE BEAT)

def get_xyz_channels(channel_names):
    aliases = {"X": {"x", "vx", "frankx", "frank_x"},
               "Y": {"y", "vy", "franky", "frank_y"},
               "Z": {"z", "vz", "frankz", "frank_z"}}
    normalized = {name: str(name).lower().replace("_", "").replace("-", "") for name in channel_names}
    res = {}
    for axis, acc in aliases.items():
        hits = [orig for orig, norm in normalized.items() if norm in acc]
        if len(hits) != 1:
            raise ValueError(f"Không tìm thấy duy nhất kênh {axis}: {hits}")
        res[axis] = hits[0]
    return res

def choose_detection_channel(channel_names):
    low = {str(n).lower(): n for n in channel_names}
    for pref in ["ii", "i", "v2"]:
        if pref in low: return low[pref]
    xyz_set = set(get_xyz_channels(channel_names).values())
    cands = [n for n in channel_names if n not in xyz_set]
    return cands[0] if cands else channel_names[0]

def load_ptb_record(record_path, max_seconds=30):
    header = wfdb.rdheader(str(record_path))
    xyz_names = get_xyz_channels(header.sig_name)
    r_name = choose_detection_channel(header.sig_name)
    selected = [xyz_names[ax] for ax in "XYZ"] + [r_name]
    sampto = min(header.sig_len, int(max_seconds * header.fs)) if max_seconds else header.sig_len
    rec = wfdb.rdrecord(str(record_path), channel_names=selected, sampto=sampto)
    cols = {name: rec.p_signal[:, i].astype(float) for i, name in enumerate(rec.sig_name)}
    xyz = np.column_stack([cols[xyz_names[ax]] for ax in "XYZ"])
    return {
        "xyz": xyz, "r_signal": cols[r_name], "r_channel": r_name,
        "fs": float(rec.fs), "header": header
    }

def safe_savgol_window(n_samples, desired_window, polyorder):
    window = min(int(desired_window), n_samples if n_samples % 2 == 1 else n_samples - 1)
    if window % 2 == 0: window -= 1
    return max(polyorder + 3 if (polyorder + 3) % 2 == 1 else polyorder + 4, window)

def preprocess_xyz(xyz, window_samples=1201, polyorder=2):
    w = safe_savgol_window(len(xyz), window_samples, polyorder)
    filtered = np.column_stack([
        savgol_filter(xyz[:, i], window_length=w, polyorder=polyorder, mode="interp")
        for i in range(3)
    ])
    return filtered

def detect_r_peaks(ecg_signal, fs):
    ecg = np.asarray(ecg_signal, dtype=float)
    nyquist = fs / 2
    sos = butter(2, [5.0 / nyquist, 20.0 / nyquist], btype="bandpass", output="sos")
    bandpassed = sosfiltfilt(sos, ecg)
    distance = int(0.3 * fs)
    prominence = 0.4 * np.ptp(bandpassed)
    cand_peaks, _ = find_peaks(bandpassed, distance=distance, prominence=prominence)
    if len(cand_peaks) == 0:
        cand_peaks, _ = find_peaks(bandpassed, distance=distance)
    refined = []
    half_window = int(0.08 * fs)
    for p in cand_peaks:
        start = max(0, p - half_window)
        end = min(len(ecg), p + half_window + 1)
        refined.append(start + np.argmax(np.abs(ecg[start:end])))
    return np.unique(refined)

def extract_median_beat(xyz, r_peaks, fs, pre_r_ms=300, post_r_ms=500, baseline_ms=100):
    pre = int(round(pre_r_ms * fs / 1000))
    post = int(round(post_r_ms * fs / 1000))
    beats = []
    for peak in r_peaks:
        start, stop = peak - pre, peak + post + 1
        if start >= 0 and stop <= len(xyz):
            beats.append(xyz[start:stop])
    if not beats:
        raise ValueError("Không cắt được nhịp nào!")
    # Nhịp đại diện trung vị (kháng ngoại lai và triệt tiêu trôi đường đẳng điện)
    rep = np.median(np.asarray(beats), axis=0)
    # Khử baseline theo đoạn PQ đầu nhịp
    baseline_n = max(1, min(len(rep), int(baseline_ms * fs / 1000)))
    rep = rep - np.median(rep[:baseline_n], axis=0)
    return rep, pre


In [ ]:
# C. THUẬT TOÁN PHÂN ĐOẠN QRS VÀ T MẠNH MẼ (ROBUST DELINEATION ENGINE)

def delineate_qrs_t_diagnostic(rep_xyz, fs, r_index):
    """
    Thuật toán phân đoạn 2 bước thích nghi kèm hàng rào sinh lý:
    1. QRS: Dựa trên Vận tốc không gian v(t) = ||d(xyz)/dt|| (bắt điểm J).
    2. T: Dựa trên Độ lớn vector M(t) = ||xyz|| (bắt bao hình sóng T).
    """
    xyz = np.asarray(rep_xyz, dtype=float)
    n = len(xyz)
    time_ms = (np.arange(n) - r_index) / fs * 1000.0
    dt = 1.0 / fs

    # 1. Tính vận tốc biến thiên không gian v(t)
    speed = np.linalg.norm(np.gradient(xyz, dt, axis=0), axis=1)
    smooth_w = safe_savgol_window(n, max(5, int(0.021 * fs) | 1), 2)
    smooth_speed = savgol_filter(speed, smooth_w, 2)

    # Tìm đỉnh vận tốc thực sự (trên sườn dốc QRS)
    q_lo = max(0, r_index - int(0.12 * fs))
    q_hi = min(n - 1, r_index + int(0.14 * fs))
    peak_speed_idx = q_lo + int(np.argmax(smooth_speed[q_lo:q_hi + 1]))

    # Tính ngưỡng vận tốc thích nghi
    base_zone = np.r_[smooth_speed[:max(1, q_lo - int(0.05 * fs))], smooth_speed[min(n, r_index + int(0.40 * fs)):]]
    q_base = float(np.median(base_zone)) if base_zone.size else float(np.median(smooth_speed))
    q_threshold = q_base + 0.12 * max(float(smooth_speed[peak_speed_idx]) - q_base, 0)

    run = max(2, int(0.012 * fs))
    # Quét lùi tìm QRS_on
    qrs_on = q_lo
    cnt = 0
    for idx in range(min(r_index, peak_speed_idx), q_lo - 1, -1):
        cnt = cnt + 1 if smooth_speed[idx] <= q_threshold else 0
        if cnt >= run:
            qrs_on = idx + run - 1
            break

    # Quét tiến tìm QRS_off (điểm thắt J-point)
    qrs_off = q_hi
    cnt = 0
    for idx in range(max(r_index, peak_speed_idx), q_hi + 1):
        cnt = cnt + 1 if smooth_speed[idx] <= q_threshold else 0
        if cnt >= run:
            qrs_off = idx - run + 1
            break

    # Hàng rào sinh lý bảo vệ cho QRS
    qrs_on = min(qrs_on, r_index - int(0.035 * fs))
    qrs_off = max(qrs_off, r_index + int(0.035 * fs))

    # 2. Tính Vector Magnitude M(t) để tách sóng T
    mag = np.linalg.norm(xyz, axis=1)
    t_lo = min(n - 2, qrs_off + int(0.03 * fs))
    t_hi = min(n - 1, r_index + int(0.48 * fs))
    t_peak = t_lo + int(np.argmax(mag[t_lo:t_hi + 1]))
    t_base = float(np.median(np.r_[mag[:max(1, qrs_on)], mag[max(t_hi, 0):]]))
    t_threshold = t_base + 0.10 * max(float(mag[t_peak]) - t_base, 0)

    t_run = max(2, int(0.015 * fs))
    t_on = t_lo
    cnt = 0
    for idx in range(t_peak, t_lo - 1, -1):
        cnt = cnt + 1 if mag[idx] <= t_threshold else 0
        if cnt >= t_run:
            t_on = idx + t_run - 1
            break

    t_off = t_hi
    cnt = 0
    for idx in range(t_peak, t_hi + 1):
        cnt = cnt + 1 if mag[idx] <= t_threshold else 0
        if cnt >= t_run:
            t_off = idx - t_run + 1
            break

    # Ràng buộc sinh lý cho T
    t_on = max(t_on, qrs_off + int(0.02 * fs))
    t_off = max(t_off, t_on + int(0.06 * fs))

    return {
        "qrs_on": qrs_on, "qrs_off": qrs_off,
        "t_on": t_on, "t_peak": t_peak, "t_off": t_off,
        "time_ms": time_ms, "speed": speed, "smooth_speed": smooth_speed,
        "q_threshold": q_threshold, "magnitude": mag,
        "t_threshold": t_threshold, "t_peak_idx": t_peak,
        "qrs_loop": xyz[qrs_on:qrs_off + 1],
        "t_loop": xyz[t_on:t_off + 1],
        "xyz": xyz, "r_index": r_index
    }


## D. Thí Nghiệm 1: Trực quan hóa Phân Đoạn & Vòng Lặp VCG Đồng Bộ (HC — patient104)

Bố cục trực quan hóa đồng bộ giữa **Miền Thời Gian** (Cột Trái) và **Không Gian Vòng Lặp VCG** (Cột Phải):
1. **Cột Trái (Miền Thời Gian):**
   * *Hàng 1 (Frank X, Y, Z):* Nhịp tim đại diện, phân tách rõ vùng QRS (xanh lá) và vùng T (cam).
   * *Hàng 2 (Vận tốc v(t)):* Bắt chính xác điểm thắt J-point (khi vận tốc rơi chạm đáy về 0) để chốt ranh giới kết thúc QRS.
   * *Hàng 3 (Vector Magnitude M(t)):* Bắt đỉnh và bao hình sóng T.
2. **Cột Phải (Miền Không Gian VCG Loop):**
   * Hai vòng lặp **QRS Loop** (xanh lá) và **T Loop** (cam) được tách rời hoàn toàn trong không gian 2D mặt phẳng Frank (X - Y), liên kết đối ứng 1-1 trực quan với vùng thời gian bên trái.


In [ ]:
# Chạy phân đoạn trên Người Khỏe Mạnh (HC)
rec_hc = load_ptb_record(DATA_ROOT / "patient104/s0306lre")
filt_hc = preprocess_xyz(rec_hc["xyz"])
rp_hc = detect_r_peaks(rec_hc["r_signal"], rec_hc["fs"])
rep_hc, r_idx_hc = extract_median_beat(filt_hc, rp_hc, rec_hc["fs"])
del_hc = delineate_qrs_t_diagnostic(rep_hc, rec_hc["fs"], r_idx_hc)

# Trực quan hóa đồng bộ: Miền Thời Gian (Trái) & Vòng Lặp Không Gian (Phải)
fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(3, 2, width_ratios=[1.25, 1.0])
t = del_hc["time_ms"]

q_on, q_off = del_hc["qrs_on"], del_hc["qrs_off"]
t_on, t_off = del_hc["t_on"], del_hc["t_off"]
qrs_pts = rep_hc[q_on:q_off + 1]
t_pts = rep_hc[t_on:t_off + 1]

# 1. Frank X, Y, Z theo thời gian
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(t, rep_hc[:, 0], label="Frank X", color="#2563eb", lw=1.5)
ax0.plot(t, rep_hc[:, 1], label="Frank Y", color="#059669", lw=1.5)
ax0.plot(t, rep_hc[:, 2], label="Frank Z", color="#f59e0b", lw=1.5)
ax0.axvspan(t[q_on], t[q_off], color="#10b981", alpha=0.25, label=f"Vòng QRS [{t[q_on]:.0f} đến {t[q_off]:.0f} ms]")
ax0.axvspan(t[t_on], t[t_off], color="#f97316", alpha=0.25, label=f"Vòng T [{t[t_on]:.0f} đến {t[t_off]:.0f} ms]")
ax0.scatter([0], [rep_hc[r_idx_hc, 1]], color="red", s=60, zorder=5, label="Đỉnh R (t = 0 ms)")
ax0.set_ylabel("Điện thế (mV)", fontweight='bold')
ax0.set_title("1. Nhịp đại diện Frank X, Y, Z — Vùng Cắt QRS & T", fontweight='bold')
ax0.legend(loc="upper right", frameon=True, ncol=2, fontsize=8.5)
ax0.grid(True, ls="--", alpha=0.4)

# 2. Vận tốc không gian v(t)
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
ax1.plot(t, del_hc["smooth_speed"], color="#4f46e5", lw=1.8, label="Vận tốc không gian v(t)")
ax1.axhline(del_hc["q_threshold"], color="red", ls="--", lw=1.2, label=f"Ngưỡng QRS Threshold ({del_hc['q_threshold']:.3f} mV/s)")
ax1.axvline(t[q_on], color="#059669", ls=":", lw=2, label=f"QRS_on ({t[q_on]:.0f} ms)")
ax1.axvline(t[q_off], color="#dc2626", ls=":", lw=2, label=f"QRS_off / J-point ({t[q_off]:.0f} ms)")
ax1.annotate("Điểm thắt J-point\n(Vận tốc rơi về đáy -> Hết QRS)", 
             xy=(t[q_off], del_hc["smooth_speed"][q_off]), 
             xytext=(t[q_off] + 25, del_hc["smooth_speed"][q_off] + 0.8),
             arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.5), fontweight="bold", fontsize=9)
ax1.set_ylabel("Vận tốc (mV/s)", fontweight='bold')
ax1.set_title("2. Động học Vận tốc v(t) dùng để bắt điểm J-point", fontweight='bold')
ax1.legend(loc="upper right", frameon=True, fontsize=8.5)
ax1.grid(True, ls="--", alpha=0.4)

# 3. Vector Magnitude M(t)
ax2 = fig.add_subplot(gs[2, 0], sharex=ax0)
ax2.plot(t, del_hc["magnitude"], color="#0891b2", lw=2.0, label="Vector Magnitude M(t)")
ax2.axhline(del_hc["t_threshold"], color="#d97706", ls="--", lw=1.2, label=f"Ngưỡng T Threshold ({del_hc['t_threshold']:.3f} mV)")
ax2.axvline(t[t_on], color="#ea580c", ls=":", lw=2, label=f"T_on ({t[t_on]:.0f} ms)")
ax2.scatter([t[del_hc["t_peak"]]], [del_hc["magnitude"][del_hc["t_peak"]]], color="#ea580c", s=70, zorder=5, label="Đỉnh T")
ax2.axvline(t[t_off], color="#7c2d12", ls=":", lw=2, label=f"T_off ({t[t_off]:.0f} ms)")
ax2.set_xlabel("Thời gian so với đỉnh R (ms)", fontweight='bold')
ax2.set_ylabel("Magnitude (mV)", fontweight='bold')
ax2.set_title("3. Độ lớn Vector M(t) dùng để tách sóng T", fontweight='bold')
ax2.legend(loc="upper right", frameon=True, fontsize=8.5)
ax2.grid(True, ls="--", alpha=0.4)

# ==========================================
# CỘT PHẢI - 2 VÒNG LẶP VCG LOOP TRONG KHÔNG GIAN
# ==========================================
ax_loop = fig.add_subplot(gs[:, 1])

# Vòng QRS Loop (Xanh lá)
ax_loop.plot(qrs_pts[:, 0], qrs_pts[:, 1], color="#10b981", lw=2.5, label="Vòng Lặp QRS (Khử cực thất)")
ax_loop.fill(qrs_pts[:, 0], qrs_pts[:, 1], color="#10b981", alpha=0.18)
ax_loop.scatter([qrs_pts[0, 0]], [qrs_pts[0, 1]], color="#059669", s=60, marker="o", label="Bắt đầu QRS (QRS_on)")
ax_loop.scatter([qrs_pts[-1, 0]], [qrs_pts[-1, 1]], color="#dc2626", s=70, marker="x", zorder=5, label="Điểm J-point (Hết QRS)")

# Vòng T Loop (Cam)
ax_loop.plot(t_pts[:, 0], t_pts[:, 1], color="#f97316", lw=2.5, ls="--", label="Vòng Lặp T (Tái cực thất)")
ax_loop.fill(t_pts[:, 0], t_pts[:, 1], color="#f97316", alpha=0.18)
ax_loop.scatter([t_pts[0, 0]], [t_pts[0, 1]], color="#ea580c", s=60, marker="^", label="Bắt đầu T (T_on)")

# Gốc đẳng điện (0, 0)
ax_loop.scatter([0], [0], color="black", s=80, marker="*", zorder=6, label="Gốc đẳng điện (0, 0)")
ax_loop.axhline(0, color="gray", lw=0.8, ls=":")
ax_loop.axvline(0, color="gray", lw=0.8, ls=":")

ax_loop.set_title("4. HAI VÒNG LẶP ĐƯỢC TÁCH RỜI TRONG KHÔNG GIAN VCG\n(Mặt phẳng Frank X - Y)", fontweight='bold', fontsize=12)
ax_loop.set_xlabel("Trục Frank X (mV)", fontweight='bold')
ax_loop.set_ylabel("Trục Frank Y (mV)", fontweight='bold')
ax_loop.legend(loc="upper left", frameon=True, fontsize=9.5)
ax_loop.grid(True, ls="--", alpha=0.5)
ax_loop.axis("equal")

plt.tight_layout()
plt.show()

print(f"-> HC Kết quả phân đoạn:")
print(f"   QRS Interval: [{t[del_hc['qrs_on']]:.0f} ms -> {t[del_hc['qrs_off']]:.0f} ms] (Thời lượng = {t[del_hc['qrs_off']] - t[del_hc['qrs_on']]:.0f} ms)")
print(f"   Sóng T Interval: [{t[del_hc['t_on']]:.0f} ms -> {t[del_hc['t_off']]:.0f} ms] (Thời lượng = {t[del_hc['t_off']] - t[del_hc['t_on']]:.0f} ms)")


## E. Thí Nghiệm 2: Trực quan hóa Delineation trên Bệnh Nhân Nhồi Máu Cơ Tim (MI — patient001)

Quan sát sự bền bỉ của thuật toán ở ca bệnh nặng:
* **Điện thế QRS bị suy giảm nặng:** chỉ còn 0.025 mV (bằng khoảng 16% so với người khỏe mạnh HC).
* **Vòng lặp không gian bị sụp đổ:** Vòng QRS Loop teo tóp, diện tích suy giảm rõ rệt.
* **Nhờ cơ chế ngưỡng thích nghi:** Thuật toán vẫn khoanh trúng vùng QRS_on, QRS_off và Sóng T mà không bị tràn hay lỗi.


In [ ]:
# Chạy phân đoạn trên Bệnh Nhân Nhồi Máu Cơ Tim (MI)
rec_mi = load_ptb_record(DATA_ROOT / "patient001/s0010_re")
filt_mi = preprocess_xyz(rec_mi["xyz"])
rp_mi = detect_r_peaks(rec_mi["r_signal"], rec_mi["fs"])
rep_mi, r_idx_mi = extract_median_beat(filt_mi, rp_mi, rec_mi["fs"])
del_mi = delineate_qrs_t_diagnostic(rep_mi, rec_mi["fs"], r_idx_mi)

# Trực quan hóa chi tiết đồng bộ cho MI
fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(3, 2, width_ratios=[1.25, 1.0])
t = del_mi["time_ms"]

q_on, q_off = del_mi["qrs_on"], del_mi["qrs_off"]
t_on, t_off = del_mi["t_on"], del_mi["t_off"]
qrs_pts = rep_mi[q_on:q_off + 1]
t_pts = rep_mi[t_on:t_off + 1]

# 1. Frank X, Y, Z cho MI
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(t, rep_mi[:, 0], label="Frank X", color="#dc2626", lw=1.5)
ax0.plot(t, rep_mi[:, 1], label="Frank Y", color="#9333ea", lw=1.5)
ax0.plot(t, rep_mi[:, 2], label="Frank Z", color="#ca8a04", lw=1.5)
ax0.axvspan(t[q_on], t[q_off], color="#10b981", alpha=0.25, label=f"Vòng QRS [{t[q_on]:.0f} đến {t[q_off]:.0f} ms]")
ax0.axvspan(t[t_on], t[t_off], color="#f97316", alpha=0.25, label=f"Vòng T [{t[t_on]:.0f} đến {t[t_off]:.0f} ms]")
ax0.scatter([0], [rep_mi[r_idx_mi, 1]], color="red", s=60, zorder=5, label="Đỉnh R (t = 0 ms)")
ax0.set_ylabel("Điện thế (mV)", fontweight='bold')
ax0.set_title("1. Nhịp đại diện Frank X, Y, Z — Nhồi Máu Cơ Tim (patient001)", fontweight='bold')
ax0.legend(loc="upper right", frameon=True, ncol=2, fontsize=8.5)
ax0.grid(True, ls="--", alpha=0.4)

# 2. Vận tốc không gian v(t) ở ca MI
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
ax1.plot(t, del_mi["smooth_speed"], color="#b91c1c", lw=1.8, label="Vận tốc không gian v(t)")
ax1.axhline(del_mi["q_threshold"], color="black", ls="--", lw=1.2, label=f"Ngưỡng QRS Threshold ({del_mi['q_threshold']:.3f} mV/s)")
ax1.axvline(t[q_on], color="#059669", ls=":", lw=2, label=f"QRS_on ({t[q_on]:.0f} ms)")
ax1.axvline(t[q_off], color="#dc2626", ls=":", lw=2, label=f"QRS_off / J-point ({t[q_off]:.0f} ms)")
ax1.set_ylabel("Vận tốc (mV/s)", fontweight='bold')
ax1.set_title("2. Động học Vận tốc v(t) ở ca MI (Vận tốc đỉnh sụp giảm trầm trọng)", fontweight='bold')
ax1.legend(loc="upper right", frameon=True, fontsize=8.5)
ax1.grid(True, ls="--", alpha=0.4)

# 3. Vector Magnitude M(t) ở ca MI
ax2 = fig.add_subplot(gs[2, 0], sharex=ax0)
ax2.plot(t, del_mi["magnitude"], color="#c026d3", lw=2.0, label="Vector Magnitude M(t)")
ax2.axhline(del_mi["t_threshold"], color="#d97706", ls="--", lw=1.2, label=f"Ngưỡng T Threshold ({del_mi['t_threshold']:.3f} mV)")
ax2.axvline(t[t_on], color="#ea580c", ls=":", lw=2, label=f"T_on ({t[t_on]:.0f} ms)")
ax2.scatter([t[del_mi["t_peak"]]], [del_mi["magnitude"][del_mi["t_peak"]]], color="#ea580c", s=70, zorder=5, label="Đỉnh T")
ax2.axvline(t[del_mi["t_off"]], color="#7c2d12", ls=":", lw=2, label=f"T_off ({t[del_mi['t_off']]:.0f} ms)")
ax2.set_xlabel("Thời gian so với đỉnh R (ms)", fontweight='bold')
ax2.set_ylabel("Magnitude (mV)", fontweight='bold')
ax2.set_title("3. Độ lớn Vector M(t) ở ca MI (Sóng T dẹt, biên độ cực thấp)", fontweight='bold')
ax2.legend(loc="upper right", frameon=True, fontsize=8.5)
ax2.grid(True, ls="--", alpha=0.4)

# Cột Phải: Vòng lặp VCG Loop ở ca MI
ax_loop = fig.add_subplot(gs[:, 1])
ax_loop.plot(qrs_pts[:, 0], qrs_pts[:, 1], color="#dc2626", lw=2.5, label="Vòng Lặp QRS (Sụp đổ diện tích)")
ax_loop.fill(qrs_pts[:, 0], qrs_pts[:, 1], color="#dc2626", alpha=0.18)
ax_loop.scatter([qrs_pts[0, 0]], [qrs_pts[0, 1]], color="#059669", s=60, marker="o", label="Bắt đầu QRS (QRS_on)")
ax_loop.scatter([qrs_pts[-1, 0]], [qrs_pts[-1, 1]], color="#dc2626", s=70, marker="x", zorder=5, label="Điểm J-point (Hết QRS)")

ax_loop.plot(t_pts[:, 0], t_pts[:, 1], color="#7c2d12", lw=2.5, ls="--", label="Vòng Lặp T (Biến dạng, teo tóp)")
ax_loop.fill(t_pts[:, 0], t_pts[:, 1], color="#b45309", alpha=0.18)
ax_loop.scatter([t_pts[0, 0]], [t_pts[0, 1]], color="#ea580c", s=60, marker="^", label="Bắt đầu T (T_on)")

ax_loop.scatter([0], [0], color="black", s=80, marker="*", zorder=6, label="Gốc đẳng điện (0, 0)")
ax_loop.axhline(0, color="gray", lw=0.8, ls=":")
ax_loop.axvline(0, color="gray", lw=0.8, ls=":")

ax_loop.set_title("4. HAI VÒNG LẶP Ở BỆNH NHÂN NHỒI MÁU CƠ TIM (MI)\n(Mặt phẳng Frank X - Y)", fontweight='bold', fontsize=12)
ax_loop.set_xlabel("Trục Frank X (mV)", fontweight='bold')
ax_loop.set_ylabel("Trục Frank Y (mV)", fontweight='bold')
ax_loop.legend(loc="upper left", frameon=True, fontsize=9.5)
ax_loop.grid(True, ls="--", alpha=0.5)
ax_loop.axis("equal")

plt.tight_layout()
plt.show()

print(f"-> MI Kết quả phân đoạn:")
print(f"   QRS Interval: [{t[del_mi['qrs_on']]:.0f} ms -> {t[del_mi['qrs_off']]:.0f} ms] (Thời lượng = {t[del_mi['qrs_off']] - t[del_mi['qrs_on']]:.0f} ms)")
print(f"   Sóng T Interval: [{t[del_mi['t_on']]:.0f} ms -> {t[del_mi['t_off']]:.0f} ms] (Thời lượng = {t[del_mi['t_off']] - t[del_mi['t_on']]:.0f} ms)")


## F. Thí Nghiệm 3: Trực quan hóa Vòng Lặp QRS và T Sau Khi Tách (3D Loops & Chiếu PCA 2D)

Sau khi tách thành công ranh giới mẫu, hai vòng lặp 3D độc lập:
* `qrs_loop` (Khử cực thất)
* `t_loop` (Tái cực thất)

được vẽ riêng biệt trong không gian 3 chiều và chiếu xuống mặt phẳng trực giao tối ưu $PC_1 - PC_2$:


In [ ]:
# Hàm chiếu PCA 2D và tính diện tích Shoelace
def project_and_calc_area(loop_3d):
    if len(loop_3d) < 3:
        return np.zeros((len(loop_3d), 2)), 0.0, 0.0, np.zeros(2), np.array([1.0, 0.0])
    pca = PCA(n_components=2)
    loop_2d = pca.fit_transform(loop_3d)
    centroid = np.mean(loop_2d, axis=0)
    x = loop_2d[:, 0]
    y = loop_2d[:, 1]
    shoelace_area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    arc_length = np.sum(np.linalg.norm(np.diff(loop_2d, axis=0), axis=1))
    return loop_2d, shoelace_area, arc_length, centroid, pca.explained_variance_ratio_

# Tính toán cho cả 2 đối tượng
qrs_2d_hc, area_q_hc, arc_q_hc, cent_q_hc, exp_q_hc = project_and_calc_area(del_hc["qrs_loop"])
t_2d_hc, area_t_hc, arc_t_hc, cent_t_hc, exp_t_hc = project_and_calc_area(del_hc["t_loop"])

qrs_2d_mi, area_q_mi, arc_q_mi, cent_q_mi, exp_q_mi = project_and_calc_area(del_mi["qrs_loop"])
t_2d_mi, area_t_mi, arc_t_mi, cent_t_mi, exp_t_mi = project_and_calc_area(del_mi["t_loop"])

# Vẽ biểu đồ 2x2: Hàng 1 = Vòng 3D, Hàng 2 = Chiếu PCA 2D
fig = plt.figure(figsize=(16, 12))

# 1. 3D Loop HC
ax_3d_hc = fig.add_subplot(2, 2, 1, projection='3d')
ax_3d_hc.plot(del_hc["qrs_loop"][:, 0], del_hc["qrs_loop"][:, 1], del_hc["qrs_loop"][:, 2], 
              color="#2563eb", lw=2.5, label="QRS Loop (3D)")
ax_3d_hc.plot(del_hc["t_loop"][:, 0], del_hc["t_loop"][:, 1], del_hc["t_loop"][:, 2], 
              color="#ea580c", lw=2.0, ls="--", label="T Loop (3D)")
ax_3d_hc.scatter([0], [0], [0], color="black", s=60, marker="o", label="Gốc (0,0,0)")
ax_3d_hc.set_title("Vòng lặp 3D Tách rời: Người Khỏe Mạnh (HC)", fontweight='bold')
ax_3d_hc.set_xlabel("Frank X (mV)")
ax_3d_hc.set_ylabel("Frank Y (mV)")
ax_3d_hc.set_zlabel("Frank Z (mV)")
ax_3d_hc.legend(loc="upper left")

# 2. 3D Loop MI
ax_3d_mi = fig.add_subplot(2, 2, 2, projection='3d')
ax_3d_mi.plot(del_mi["qrs_loop"][:, 0], del_mi["qrs_loop"][:, 1], del_mi["qrs_loop"][:, 2], 
              color="#dc2626", lw=2.5, label="QRS Loop (3D)")
ax_3d_mi.plot(del_mi["t_loop"][:, 0], del_mi["t_loop"][:, 1], del_mi["t_loop"][:, 2], 
              color="#7c2d12", lw=2.0, ls="--", label="T Loop (3D)")
ax_3d_mi.scatter([0], [0], [0], color="black", s=60, marker="o", label="Gốc (0,0,0)")
ax_3d_mi.set_title("Vòng lặp 3D Tách rời: Bệnh Nhân MI (Co cụm, sụt áp)", fontweight='bold')
ax_3d_mi.set_xlabel("Frank X (mV)")
ax_3d_mi.set_ylabel("Frank Y (mV)")
ax_3d_mi.set_zlabel("Frank Z (mV)")
ax_3d_mi.legend(loc="upper left")

# 3. PCA 2D Projection HC
ax_pca_hc = fig.add_subplot(2, 2, 3)
ax_pca_hc.plot(qrs_2d_hc[:, 0], qrs_2d_hc[:, 1], color="#2563eb", lw=2.2, label=f"QRS (Area = {area_q_hc:.6f})")
ax_pca_hc.fill(qrs_2d_hc[:, 0], qrs_2d_hc[:, 1], color="#3b82f6", alpha=0.2)
ax_pca_hc.scatter([cent_q_hc[0]], [cent_q_hc[1]], color="red", s=50, label="QRS Centroid")
ax_pca_hc.plot(t_2d_hc[:, 0], t_2d_hc[:, 1], color="#ea580c", lw=1.8, ls="--", label=f"T Loop (Area = {area_t_hc:.6f})")
ax_pca_hc.fill(t_2d_hc[:, 0], t_2d_hc[:, 1], color="#fb923c", alpha=0.15)
ax_pca_hc.set_title(f"Chiếu PCA 2D (HC): Vòng elip mở rộng căng tràn\nPhương sai giữ lại: QRS={(exp_q_hc.sum()*100):.1f}%", fontweight='bold')
ax_pca_hc.set_xlabel("PC1 (mV)")
ax_pca_hc.set_ylabel("PC2 (mV)")
ax_pca_hc.legend(loc="best")
ax_pca_hc.axis("equal")

# 4. PCA 2D Projection MI
ax_pca_mi = fig.add_subplot(2, 2, 4)
ax_pca_mi.plot(qrs_2d_mi[:, 0], qrs_2d_mi[:, 1], color="#dc2626", lw=2.2, label=f"QRS (Area = {area_q_mi:.8f})")
ax_pca_mi.fill(qrs_2d_mi[:, 0], qrs_2d_mi[:, 1], color="#ef4444", alpha=0.2)
ax_pca_mi.scatter([cent_q_mi[0]], [cent_q_mi[1]], color="black", s=50, label="QRS Centroid")
ax_pca_mi.plot(t_2d_mi[:, 0], t_2d_mi[:, 1], color="#7c2d12", lw=1.8, ls="--", label=f"T Loop (Area = {area_t_mi:.6f})")
ax_pca_mi.fill(t_2d_mi[:, 0], t_2d_mi[:, 1], color="#b45309", alpha=0.15)
ax_pca_mi.set_title(f"Chiếu PCA 2D (MI): Diện tích sụp đổ teo tóp\nPhương sai giữ lại: QRS={(exp_q_mi.sum()*100):.1f}%", fontweight='bold')
ax_pca_mi.set_xlabel("PC1 (mV)")
ax_pca_mi.set_ylabel("PC2 (mV)")
ax_pca_mi.legend(loc="best")
ax_pca_mi.axis("equal")

plt.tight_layout()
plt.show()


## G. Thí Nghiệm 4: Bảng Định Lượng 12 Đặc Trưng Sau Khi Tách Loop

Bằng việc phân tách thành công hai vòng lặp QRS và T, ta tính toán trọn vẹn 12 đặc trưng toán học chuẩn của bài báo *Vondrák & Penhaker (2026)*:


In [ ]:
# Tính toán trọn vẹn 12 đặc trưng
def extract_12_features(del_result, fs):
    qrs = del_result["qrs_loop"]
    t = del_result["t_loop"]
    
    q_2d, area_q, arc_q, _, _ = project_and_calc_area(qrs)
    t_2d, area_t, arc_t, _, _ = project_and_calc_area(t)
    
    dt = 1.0 / fs
    q_vel = np.linalg.norm(np.gradient(qrs, dt, axis=0), axis=1)
    t_vel = np.linalg.norm(np.gradient(t, dt, axis=0), axis=1)
    
    q_cent = np.mean(qrs, axis=0)
    
    return {
        "QRS_Arc": arc_q,
        "QRS_Area": area_q,
        "QRS_MaxVec": float(np.linalg.norm(qrs, axis=1).max()),
        "QRS_MaxCentroidDistance": float(np.linalg.norm(qrs - q_cent, axis=1).max()),
        "QRS_MaxVel": float(q_vel.max()),
        "QRS_MeanVel": float(q_vel.mean()),
        "QRS_StdVel": float(q_vel.std()),
        "T_MaxVec": float(np.linalg.norm(t, axis=1).max()),
        "T_Area": area_t,
        "T_MaxVel": float(t_vel.max()),
        "T_MeanVel": float(t_vel.mean()),
        "T_StdVel": float(t_vel.std()),
    }

feats_hc = extract_12_features(del_hc, rec_hc["fs"])
feats_mi = extract_12_features(del_mi, rec_mi["fs"])

df_features = pd.DataFrame({
    "Healthy Control (HC)": feats_hc,
    "Myocardial Infarction (MI)": feats_mi,
})
df_features["Tỷ lệ MI / HC (%)"] = (df_features["Myocardial Infarction (MI)"] / df_features["Healthy Control (HC)"]) * 100.0
df_features["Đánh giá biến đổi"] = df_features["Tỷ lệ MI / HC (%)"].apply(
    lambda p: f"Giảm {(100 - p):.1f}%" if p < 100 else f"Tăng {(p - 100):.1f}%"
)

display(df_features.style.format({
    "Healthy Control (HC)": "{:.6f}",
    "Myocardial Infarction (MI)": "{:.6f}",
    "Tỷ lệ MI / HC (%)": "{:.2f}%"
}).set_caption("BẢNG ĐỐI SÁNH 12 ĐẶC TRƯNG HÌNH HỌC VÀ ĐỘNG HỌC VCG"))


## H. Thử Nghiệm Tự Do Với Bất Kỳ Bệnh Nhân Nào Khác Trong Dataset

Nhóm có thể chạy thử hàm `test_patient_delineation()` với bất kỳ `patient_id` và `record_id` nào trong PTB để kiểm tra tính tổng quát của thuật toán:


In [ ]:
def test_patient_delineation(patient_id="patient002", record_id="s0015lre"):
    rec_path = DATA_ROOT / f"{patient_id}/{record_id}"
    if not rec_path.with_suffix(".hea").exists():
        print(f"Lỗi: Không tìm thấy file {rec_path}")
        return
    
    raw = load_ptb_record(rec_path)
    filt = preprocess_xyz(raw["xyz"])
    rp = detect_r_peaks(raw["r_signal"], raw["fs"])
    rep, r_idx = extract_median_beat(filt, rp, raw["fs"])
    res = delineate_qrs_t_diagnostic(rep, raw["fs"], r_idx)
    
    t = res["time_ms"]
    q_dur = t[res["qrs_off"]] - t[res["qrs_on"]]
    t_dur = t[res["t_off"]] - t[res["t_on"]]
    
    print(f"=== KẾT QUẢ PHÂN ĐOẠN: {patient_id} ({record_id}) ===")
    print(f"Chẩn đoán hồ sơ: {raw['header'].comments[0] if raw['header'].comments else 'N/A'}")
    print(f"-> QRS Interval: [{t[res['qrs_on']]:.0f} ms -> {t[res['qrs_off']]:.0f} ms] | Thời lượng: {q_dur:.0f} ms")
    print(f"-> T Loop Interval: [{t[res['t_on']]:.0f} ms -> {t[res['t_off']]:.0f} ms] | Thời lượng: {t_dur:.0f} ms")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(t, res["magnitude"], lw=1.8, color="#2563eb", label="Vector Magnitude M(t)")
    ax1.axvspan(t[res["qrs_on"]], t[res["qrs_off"]], color="#10b981", alpha=0.3, label="QRS Loop")
    ax1.axvspan(t[res["t_on"]], t[res["t_off"]], color="#f97316", alpha=0.3, label="T Loop")
    ax1.set_title(f"Phân đoạn M(t) — {patient_id}", fontweight='bold')
    ax1.set_xlabel("Thời gian so với R (ms)")
    ax1.set_ylabel("mV")
    ax1.legend()
    
    q_2d, area_q, _, _, _ = project_and_calc_area(res["qrs_loop"])
    t_2d, area_t, _, _, _ = project_and_calc_area(res["t_loop"])
    ax2.plot(q_2d[:, 0], q_2d[:, 1], color="#10b981", lw=2, label=f"QRS Loop (Area={area_q:.6f})")
    ax2.fill(q_2d[:, 0], q_2d[:, 1], color="#10b981", alpha=0.2)
    ax2.plot(t_2d[:, 0], t_2d[:, 1], color="#f97316", lw=1.8, ls="--", label=f"T Loop (Area={area_t:.6f})")
    ax2.fill(t_2d[:, 0], t_2d[:, 1], color="#f97316", alpha=0.2)
    ax2.set_title("Vòng lặp QRS & T trên mặt phẳng PCA 2D", fontweight='bold')
    ax2.axis("equal")
    ax2.legend()
    plt.tight_layout()
    plt.show()

# Chạy thử nghiệm trên patient002
test_patient_delineation("patient002", "s0015lre")
